# Week 3 — Search Intelligence Data Contract (`Lane 1: Content Refresh`)

## 1. The Data Contract (5 Plain-Words Answers)

1. **What one row means (Granularity / Unit of Analysis):**
   In the daily fact table, one row represents daily observed search and analytics performance for one content item belonging to one client (`report_date`, `client_id`, `content_id`). At the decision layer, we aggregate a trailing pre-decision window per `(client_id, content_id)` to score each URL for content refresh.

2. **Which table(s) we use:**
   * Primary panel: `fact_content_daily_performance` (partitioned by `month=YYYY-MM` on `hf://datasets/FlyRank/internship-warehouse`).
   * Context dimensions: `dim_clients` (for `gsc_data_start` / `ga4_data_start` panel coverage) and `dim_content` (for content metadata).

3. **Time window:**
   * **Mid-panel development & verification slice:** `month=2026-03` (`2026-03-01` through `2026-03-31`) for contract verification and leakage-free feature engineering.
   * **Sealed evaluation slice:** `month=2026-06` (held out strictly for final testing).

4. **What we predict / rank (Target / Proxy):**
   * **Task:** Priority ranking / binary classification (`is_future_decay`) predicting whether a visible content item's average daily impressions in the forward window (`days 22–30` of the month) drop by $\ge 20\%$ relative to its pre-decision baseline window (`days 1–21`).
   * **Action supported:** Ranking recoverable decaying URLs into weekly editorial refresh queues (`Urgent Refresh`, `Standard Refresh`, `Review for Retirement`, `Monitor`).

5. **Deliberate exclusions:**
   * **Post-cutoff future metrics (`days 22–30`) and label-derived ratios:** Strictly excluded from features to prevent temporal leakage.
   * **Raw client domains, full URLs, and unhashed query strings:** Excluded to protect client confidentiality and prevent domain-level memorization.

In [16]:
import os
import getpass
import duckdb
import numpy as np
import pandas as pd

# Obtém HF_TOKEN do ambiente/Colab Secrets ou solicita de forma segura (nunca hardcode no notebook!)
HF_TOKEN = os.getenv("HF_TOKEN")
if not HF_TOKEN:
    try:
        from google.colab import userdata
        HF_TOKEN = userdata.get("HF_TOKEN")
    except Exception:
        HF_TOKEN = getpass.getpass("Cole seu Hugging Face READ token (HF_TOKEN): ")

assert HF_TOKEN and len(HF_TOKEN.strip()) > 10, "HF_TOKEN inválido ou ausente."
os.environ["HF_TOKEN"] = HF_TOKEN.strip()

con = duckdb.connect(database=":memory:")
con.execute(f"CREATE OR REPLACE SECRET hf_secret (TYPE huggingface, TOKEN '{HF_TOKEN.strip()}');")

MONTH = "2026-03"
BASE_URI = "hf://datasets/FlyRank/internship-warehouse"
FACT_PATH = f"{BASE_URI}/fact_content_daily_performance/month={MONTH}/*.parquet"
DIM_CLIENTS_PATH = f"{BASE_URI}/dim_clients/*.parquet"

print(f"Sessão DuckDB autenticada com sucesso. Partição alvo: month={MONTH}")

Sessão DuckDB autenticada com sucesso. Partição alvo: month=2026-03


## 2. Three Verification Queries (`month=2026-03`)

We verify three structural facts on the mid-panel partition before building any features:
1. **Grain check:** Confirm `(report_date, client_id, content_id)` has zero duplicate rows (`HAVING COUNT(*) > 1` returns empty).
2. **Row count & date bounds:** Verify exact row volume, active client/content counts, and strict calendar bounds (`2026-03-01` to `2026-03-31`).
3. **Availability survival rate:** Check how many rows survive `WHERE gsc_data_available IS TRUE` (and `ga4_data_available IS TRUE`) so zero-filled pre-onboarding rows are never treated as true zero traffic.

In [17]:
# Inspeciona rapidamente as colunas reais da tabela para confirmar o schema exato
cols_df = con.execute(f"DESCRIBE SELECT * FROM read_parquet('{FACT_PATH}') LIMIT 1;").df()
print("=== Colunas disponíveis no warehouse ===")
print(cols_df[["column_name", "column_type"]].to_string(index=False))
print()

# Descobre dinamicamente o nome exato da coluna booleana de disponibilidade GSC
avail_col = "gsc_data_available" if "gsc_data_available" in cols_df["column_name"].values else "client_has_gsc"
print(f"Usando coluna de disponibilidade: {avail_col}\n")

# Fact 1: Verificação do grão (report_date, client_hash_id, content_hash_id)
q_grain = f"""
SELECT
    report_date,
    client_hash_id,
    content_hash_id,
    COUNT(*) AS dup_count
FROM read_parquet('{FACT_PATH}')
GROUP BY report_date, client_hash_id, content_hash_id
HAVING COUNT(*) > 1
LIMIT 5;
"""
grain_dups = con.execute(q_grain).df()
print("=== Fact 1: Grain Uniqueness Check ===")
print(f"Duplicatas encontradas: {len(grain_dups)}")
assert len(grain_dups) == 0, "ERRO DE CONTRATO: Grão duplicado!"
print("STATUS: PASSED — 1 linha = exatamente 1 (report_date, client_hash_id, content_hash_id).\n")

# Fact 2: Contagem de linhas e intervalo de datas da fatia
q_span = f"""
SELECT
    COUNT(*) AS total_rows,
    MIN(report_date) AS min_date,
    MAX(report_date) AS max_date,
    COUNT(DISTINCT client_hash_id) AS n_clients,
    COUNT(DISTINCT content_hash_id) AS n_contents
FROM read_parquet('{FACT_PATH}');
"""
span_df = con.execute(q_span).df()
print("=== Fact 2: Row Count & Date Span ===")
print(span_df.to_string(index=False))
print()

# Fact 3: Taxa de sobrevivência do filtro de disponibilidade (IS TRUE)
q_avail = f"""
SELECT
    COUNT(*) AS gross_rows,
    COUNT(*) FILTER (WHERE {avail_col} IS TRUE) AS available_rows,
    ROUND(100.0 * COUNT(*) FILTER (WHERE {avail_col} IS TRUE) / COUNT(*), 2) AS survival_rate_pct
FROM read_parquet('{FACT_PATH}');
"""
avail_df = con.execute(q_avail).df()
print(f"=== Fact 3: Availability Filter (`{avail_col} IS TRUE`) ===")
print(avail_df.to_string(index=False))

=== Colunas disponíveis no warehouse ===
             column_name column_type
             report_date        DATE
          client_hash_id     VARCHAR
         content_hash_id     VARCHAR
          client_has_gsc     BOOLEAN
          client_has_ga4     BOOLEAN
      gsc_data_available     BOOLEAN
      ga4_data_available     BOOLEAN
         gsc_impressions      BIGINT
              gsc_clicks      BIGINT
        gsc_sum_position      BIGINT
        gsc_avg_position      DOUBLE
           ga4_pageviews      BIGINT
            ga4_sessions      BIGINT
               ga4_users      BIGINT
    ga4_engaged_sessions      BIGINT
ga4_total_engagement_sec      BIGINT
        sessions_organic      BIGINT
         sessions_direct      BIGINT
       sessions_referral      BIGINT
         sessions_social      BIGINT
           sessions_paid      BIGINT
             sessions_ai      BIGINT
              ai_chatgpt      BIGINT
           ai_perplexity      BIGINT
               ai_gemini      BIGI

## 3. Five Pre-Decision Features & The Deliberate Leakage Trap

To prevent temporal leakage inside `month=2026-03`, we split the month at a strict decision cutoff (`2026-03-21`):
* **Observation Window (`2026-03-01` to `2026-03-21` — 21 days):** All 5 features are computed strictly from rows on or before `2026-03-21` where `gsc_data_available IS TRUE`.
* **Outcome Window (`2026-03-22` to `2026-03-31` — 10 days):** Used *only* to compute the observed ground-truth target `is_future_decay` ($\ge 20\%$ drop in mean daily impressions vs. baseline).

### Feature Dictionary ("Available When?")
1. **`pre_impressions_mean`** (`AVG(gsc_impressions)` over days 1–21): *Knowable at the decision moment because* it summarizes historical Search Console impressions already synced prior to cutoff `2026-03-21`.
2. **`pre_clicks_sum`** (`SUM(gsc_clicks)` over days 1–21): *Knowable at the decision moment because* cumulative clicks up to the cutoff date are recorded before editorial triage happens.
3. **`pre_avg_position`** (`AVG(NULLIF(gsc_avg_position, 0))` over days 1–21): *Knowable at the decision moment because* historical SERP ranking positions (excluding `0 = no data`) are observed prior to cutoff.
4. **`pre_ctr_pct`** (`100.0 * SUM(gsc_clicks) / NULLIF(SUM(gsc_impressions), 0)` over days 1–21): *Knowable at the decision moment because* it is computed solely from pre-cutoff clicks and impressions.
5. **`pre_early_vs_late_ratio`** (ratio of mean impressions in days 15–21 vs. days 1–14): *Knowable at the decision moment because* both sub-windows precede `2026-03-21`, capturing pre-decision momentum without touching future outcomes.

In [18]:
from sklearn.model_selection import GroupShuffleSplit
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import average_precision_score, roc_auc_score

# Query que constrói as 5 features pré-corte (<= 2026-03-21) e o alvo pós-corte (> 2026-03-21)
q_model_frame = f"""
WITH panel AS (
    SELECT
        report_date,
        client_hash_id,
        content_hash_id,
        gsc_impressions,
        gsc_clicks,
        NULLIF(gsc_avg_position, 0) AS clean_position
    FROM read_parquet('{FACT_PATH}')
    WHERE {avail_col} IS TRUE
),
agg AS (
    SELECT
        client_hash_id,
        content_hash_id,
        -- 5 Features estritamente pré-decisão (01/03 a 21/03)
        AVG(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-21') AS pre_impressions_mean,
        SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-21') AS pre_clicks_sum,
        AVG(clean_position) FILTER (WHERE report_date <= DATE '2026-03-21') AS pre_avg_position,
        COALESCE(
            100.0 * SUM(gsc_clicks) FILTER (WHERE report_date <= DATE '2026-03-21')
            / NULLIF(SUM(gsc_impressions) FILTER (WHERE report_date <= DATE '2026-03-21'), 0),
            0.0
        ) AS pre_ctr_pct,
        COALESCE(
            AVG(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-15' AND DATE '2026-03-21')
            / NULLIF(AVG(gsc_impressions) FILTER (WHERE report_date BETWEEN DATE '2026-03-01' AND DATE '2026-03-14'), 0),
            1.0
        ) AS pre_early_vs_late_ratio,

        -- Janela futura (22/03 a 31/03) usada APENAS para medir o desfecho real
        AVG(gsc_impressions) FILTER (WHERE report_date > DATE '2026-03-21') AS post_impressions_mean
    FROM panel
    GROUP BY client_hash_id, content_hash_id
)
SELECT
    client_hash_id,
    content_hash_id,
    pre_impressions_mean,
    pre_clicks_sum,
    pre_avg_position,
    pre_ctr_pct,
    pre_early_vs_late_ratio,
    -- Coluna de vazamento deliberado (usa post_impressions_mean do futuro)
    (post_impressions_mean - pre_impressions_mean) / NULLIF(pre_impressions_mean, 0) AS leaked_future_pct_change,
    -- Alvo observado: queda >= 20% na janela futura
    CASE
        WHEN post_impressions_mean <= 0.80 * pre_impressions_mean THEN 1
        ELSE 0
    END AS is_future_decay
FROM agg
WHERE pre_impressions_mean >= 10.0
  AND post_impressions_mean IS NOT NULL;
"""

df_contract = con.execute(q_model_frame).df()
df_contract["pre_avg_position"] = df_contract["pre_avg_position"].fillna(df_contract["pre_avg_position"].median())

print(f"Frame de modelagem construído: {len(df_contract):,} URLs elegíveis | {df_contract['client_hash_id'].nunique()} clientes")
print(f"Taxa base de queda futura (is_future_decay == 1): {df_contract['is_future_decay'].mean():.3f}\n")

honest_features = [
    "pre_impressions_mean",
    "pre_clicks_sum",
    "pre_avg_position",
    "pre_ctr_pct",
    "pre_early_vs_late_ratio",
]

# Split agrupado por cliente (sem vazamento entre clientes de treino e teste)
gss = GroupShuffleSplit(n_splits=1, test_size=0.25, random_state=42)
train_idx, test_idx = next(gss.split(df_contract, df_contract["is_future_decay"], groups=df_contract["client_hash_id"]))

train_df = df_contract.iloc[train_idx]
test_df = df_contract.iloc[test_idx]
y_train = train_df["is_future_decay"].values
y_test = test_df["is_future_decay"].values

# 1. EXPERIMENTO COM ARMADILHA DE VAZAMENTO (feature derivada do futuro)
leaky_features = honest_features + ["leaked_future_pct_change"]
rf_leaky = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_leaky.fit(train_df[leaky_features], y_train)
preds_leaky = rf_leaky.predict_proba(test_df[leaky_features])[:, 1]

print("=== [1] LEAKAGE TRAP (Com feature derivada do futuro) ===")
print(f"PR-AUC (Vazado):  {average_precision_score(y_test, preds_leaky):.4f}  <-- Artificiosamente perfeito!")
print(f"ROC-AUC (Vazado): {roc_auc_score(y_test, preds_leaky):.4f}\n")

# 2. REMOÇÃO DA COLUNA VAZADA E AVALIAÇÃO HONESTA
df_contract = df_contract.drop(columns=["leaked_future_pct_change"])
assert "leaked_future_pct_change" not in df_contract.columns

rf_honest = RandomForestClassifier(n_estimators=50, max_depth=6, random_state=42, n_jobs=-1)
rf_honest.fit(train_df[honest_features], y_train)
preds_honest = rf_honest.predict_proba(test_df[honest_features])[:, 1]

print("=== [2] HONEST MODEL (Após deletar a coluna vazada) ===")
print(f"PR-AUC (Honesto):  {average_precision_score(y_test, preds_honest):.4f}  <-- Sinal preditivo real em clientes inéditos")
print(f"ROC-AUC (Honesto): {roc_auc_score(y_test, preds_honest):.4f}")

Frame de modelagem construído: 74,962 URLs elegíveis | 39 clientes
Taxa base de queda futura (is_future_decay == 1): 0.316

=== [1] LEAKAGE TRAP (Com feature derivada do futuro) ===
PR-AUC (Vazado):  1.0000  <-- Artificiosamente perfeito!
ROC-AUC (Vazado): 1.0000

=== [2] HONEST MODEL (Após deletar a coluna vazada) ===
PR-AUC (Honesto):  0.7045  <-- Sinal preditivo real em clientes inéditos
ROC-AUC (Honesto): 0.7979


## 4. Named Limitation of This Slice

* **Single-Month Boundary Truncation & Short-Horizon Noise:**
  Evaluating a 21-day observation window against a 10-day outcome window inside a single calendar partition (`month=2026-03`) captures short-term impression drops but cannot distinguish true structural content decay from weekly seasonality or client-level tracking onboarding (`gsc_data_start` mid-month). In the full capstone pipeline, we address this by querying multi-month rolling windows (`90d` baseline vs. `30d` forward window) joined with `dim_clients` eligibility filters.


## 5. Self-check

- [x] Every section above is filled — markdown rationale AND executable DuckDB/Python code
- [x] The notebook runs top to bottom with no errors
- [x] No client names, URLs, or unhashed queries are exposed
- [x] Claims use objective terminology: *measured*, *observed*, *directional*, *decision-time knowable*
- [x] Committed to repo under `work/notebooks/w03_data_contract.ipynb`